# Hooks System

Hooks let external scripts react to the agent's lifecycle without modifying the agent's code. A *hook* is a shell command (or an inline bash script) that runs synchronously at one of five well-defined points: before the agent starts on a message, after it finishes, before and after every tool call, and whenever an exception bubbles out of the loop.

Use cases are the usual CI-style ones: log every tool call to a file, run a linter after the agent edits source files, post a desktop notification when a long task completes, dump a stack trace to an error log, or sync a session checkpoint. The agent itself stays unaware of what the hooks do — it just calls into `HookSystem` at the five trigger points and moves on. Hook errors are swallowed (logged at WARNING) so a misconfigured hook can never break a working session.

This notebook implements the hook system from the upstream `ai-coding-agent` project and wires it into the `Agent` class we built in [NB03](/notebooks/apps/cda/03-agent.html).

## Defining a Hook: `HookConfig`

A hook is a Pydantic model with a name, a trigger, and exactly one of `command` or `script`. The two are mutually exclusive — `command` runs an existing program (e.g. `"python3 tests.py"`); `script` is an inline bash snippet that the runtime writes to a temp file with a `#!/bin/bash` shebang. `timeout_sec` caps how long the hook may run; `enabled` lets individual hooks be toggled without deleting them from the config.

In [ ]:
from notebooks.agent import HookConfig, HookTrigger

log_hook = HookConfig(
    name="pre-tool-log",
    trigger=HookTrigger.BEFORE_TOOL,
    command="echo $AI_AGENT_TOOL_NAME >> .ai-agent/tool.log",
    timeout_sec=5,
)
print(log_hook.model_dump_json(indent=2))

**The five triggers** map directly onto lifecycle points in the agent loop:

| `HookTrigger` | Fired by `Agent` at | Env vars added |
|---|---|---|
| `BEFORE_AGENT` | start of `run()`, after the user message is recorded | `AI_AGENT_USER_MESSAGE` |
| `AFTER_AGENT` | end of `run()`, before the `agent_end` event | `AI_AGENT_RESPONSE` |
| `BEFORE_TOOL` | just before each tool invocation | `AI_AGENT_TOOL_NAME`, `AI_AGENT_TOOL_PARAMS` |
| `AFTER_TOOL` | just after each tool returns | `+ AI_AGENT_TOOL_RESULT` |
| `ON_ERROR` | when an exception escapes `_agentic_loop` | `AI_AGENT_ERROR` |

: {tbl-colwidths="[20,40,40]"}

Every hook also receives `AI_AGENT_TRIGGER` (the trigger name) and `AI_AGENT_CWD` (the agent's working directory). Hooks read context from the environment rather than argv — this keeps the command line short and lets the same shell snippet work across triggers.

**Validation.** `HookConfig` rejects a hook that has neither `command` nor `script`, or has both at once:

In [ ]:
from pydantic import ValidationError

for bad in [
    {"name": "empty",     "trigger": "before_agent"},                              # neither
    {"name": "both",      "trigger": "before_agent", "command": "echo a", "script": "echo b"},  # both
    {"name": "bad-trigger", "trigger": "not_a_trigger", "command": "echo a"},    # bad enum
]:
    try:
        HookConfig(**bad)
        print(f"{bad['name']}: accepted (unexpected!)")
    except ValidationError as e:
        print(f"{bad['name']}: rejected -> {e.errors()[0]['msg']}")

## `HookSystem`: a No-Op When Disabled


The runtime side is the `HookSystem` class. Construction is cheap and stateless: it filters the config's hooks down to the enabled ones — unless `config.hooks_enabled` is false, in which case the list is empty and every trigger method is a no-op. This is the safety contract that lets the `Agent` call `self.hooks.trigger_*` unconditionally without paying any cost in the common case where no hooks are configured.

In [ ]:
from notebooks.agent import Config, HookSystem

# Default Config has hooks_enabled=False -> HookSystem is a no-op.
cfg = Config()
hs = HookSystem(cfg)
print(f"hooks_enabled = {cfg.hooks_enabled}")
print(f"active hooks  = {hs.hooks}")

import asyncio
asyncio.run(hs.trigger_before_agent("hello"))   # returns immediately, runs nothing
print("no-op trigger returned without error")

Flip `hooks_enabled` to `True` and the same hooks that were inert become live:

In [ ]:
cfg = Config(
    hooks_enabled=True,
    hooks=[
        HookConfig(name="a", trigger=HookTrigger.BEFORE_AGENT, command="echo a"),
        HookConfig(name="b", trigger=HookTrigger.AFTER_TOOL, command="echo b",
                   enabled=False),   # individually disabled
    ],
)
hs = HookSystem(cfg)
print(f"active hooks: {[(h.name, h.trigger.value) for h in hs.hooks]}")

**Two-level gating.** `hooks_enabled` is the global kill-switch — turn it off and no hooks run, period. Per-hook `enabled` is the fine-grained switch — useful for keeping a noisy logging hook defined in the config but turning it off during a focused session. The filtered list `hs.hooks` is computed once at construction; trigger methods iterate over it without re-checking on every call.

## Running a Command Hook


When a `BEFORE_AGENT` hook fires, `HookSystem` builds the environment, finds every hook whose `trigger` matches, and awaits each one. A `command` is passed straight to `asyncio.create_subprocess_shell`; an inline `script` is written to a temp file with a shebang first. We observe both by hooking a script that dumps the env it received:

In [ ]:
import asyncio, tempfile, os
from pathlib import Path

async def demo():
    with tempfile.TemporaryDirectory() as d:
        cwd = Path(d)
        env_dump = cwd / "env.txt"
        # Inline script: dump only the AI_AGENT_* vars this hook sees.
        hook = HookConfig(
            name="env-dump",
            trigger=HookTrigger.BEFORE_AGENT,
            script=f"env | grep ^AI_AGENT_ | sort > {env_dump}",
        )
        cfg = Config(cwd=cwd, hooks_enabled=True, hooks=[hook])
        hs = HookSystem(cfg)

        await hs.trigger_before_agent("Please fix the login bug.")
        print(open(env_dump).read())

asyncio.run(demo())

**Environment variables observed.** The hook saw three `AI_AGENT_*` vars:

- `AI_AGENT_TRIGGER=before_agent` — which trigger fired.
- `AI_AGENT_CWD=<tmpdir>` — the agent's working directory, also the subprocess `cwd`.
- `AI_AGENT_USER_MESSAGE=Please fix the login bug.` — the user's message verbatim.

The same env shape holds for every trigger; only the additional vars differ (see the table above). Hooks that need the user message read `$AI_AGENT_USER_MESSAGE`; hooks that need tool parameters read `$AI_AGENT_TOOL_PARAMS` (JSON-encoded).

## All Five Triggers


Each public `trigger_*` method is the same shape: build env, iterate matching hooks, await. We exercise all five against a tiny recording hook that appends one line per firing, to see the order in which the `Agent` calls them during a real run.

In [ ]:
import tempfile, asyncio
from pathlib import Path
from notebooks.agent.tools.base import ToolResult

async def all_triggers():
    with tempfile.TemporaryDirectory() as d:
        cwd = Path(d)
        log = cwd / "trace.log"
        # One hook per trigger, each appending its trigger name.
        hooks = [
            HookConfig(name=t.value, trigger=t,
                       command=f"echo {t.value} >> {log}")
            for t in HookTrigger
        ]
        cfg = Config(cwd=cwd, hooks_enabled=True, hooks=hooks)
        hs = HookSystem(cfg)

        await hs.trigger_before_agent("hi")
        await hs.trigger_before_tool("read_file", {"path": "a.py"})
        await hs.trigger_after_tool(
            "read_file", {"path": "a.py"},
            ToolResult(success=True, output="contents of a.py")
        )
        await hs.trigger_after_agent("hi", "done")
        try:
            raise RuntimeError("simulated failure")
        except RuntimeError as e:
            await hs.trigger_on_error(e)

        print(open(log).read())

asyncio.run(all_triggers())

**All five triggers fired in the order `HookSystem` exposes them.** `AFTER_TOOL` ran after `BEFORE_TOOL` (as the names suggest), and `ON_ERROR` ran last because in a real run it is called from the `except` clause wrapping `_agentic_loop`. In the agent loop proper, `BEFORE_AGENT` and `AFTER_AGENT` bracket the *whole* run, while the tool triggers bracket each individual tool invocation.

## Wiring Into the Agent Loop

The `Agent` already calls `HookSystem` at the right points — we wired the triggers in when we added the module. Three sites matter, each corresponding to a section of `Agent.run()` / `Agent._agentic_loop()`:

In [ ]:
import inspect
from notebooks.agent import Agent

src = inspect.getsource(Agent)
for line in src.splitlines():
    if "hooks.trigger" in line:
        print(line.strip())

**Five trigger calls, in the order the loop executes them:**

1. `trigger_before_agent(user_message)` — first line inside `run()`, after the user message is added to the session.
2. `trigger_before_tool(name, arguments)` — wraps each tool invocation in `_agentic_loop`.
3. `trigger_after_tool(name, arguments, result)` — same site, after `registry.invoke`.
4. `trigger_after_agent(user_message, response)` — after `_agentic_loop` returns cleanly, before the `agent_end` event is emitted to the stream consumer. The final assistant text is passed as `response` (or `""` if the run produced no text).
5. `trigger_on_error(exception)` — in the `except Exception` clause, before the `agent_error` event is yielded. This lets an `ON_ERROR` hook log the failure even when the consumer never sees the error event (e.g., if it has already stopped iterating).

:::{.callout-note}
Hooks that *fire during a tool call* block the agent loop until they return or hit `timeout_sec`. There is no background queue and no parallelism across hooks for the same trigger. This keeps the order of side effects deterministic and matches the upstream design; for fire-and-forget work, have the hook itself background a process with `&` and return.

:::

## A Practical Hook: Auto-Lint After Edits

A common real use of `AFTER_TOOL`: run a linter every time the agent edits a source file. The hook inspects `$AI_AGENT_TOOL_NAME`, and only acts when the tool was `write_file` or `edit_file`. We build the config in Python (it could equally live in `config.toml` under `[[hooks]]`):

In [ ]:
import tempfile, asyncio
from pathlib import Path

async def lint_hook_demo():
    with tempfile.TemporaryDirectory() as d:
        cwd = Path(d)
        trace = cwd / "lint.log"
        # Only lint when the edited file looks like Python.
        script = (
            'if [ "$AI_AGENT_TOOL_NAME" = "write_file" ] || '
            '[ "$AI_AGENT_TOOL_NAME" = "edit_file" ]; then\n'
            '  echo "linting after $AI_AGENT_TOOL_NAME" >> ' + str(trace) + '\n'
            'fi'
        )
        hook = HookConfig(
            name="auto-lint",
            trigger=HookTrigger.AFTER_TOOL,
            script=script,
            timeout_sec=10,
        )
        cfg = Config(cwd=cwd, hooks_enabled=True, hooks=[hook])
        hs = HookSystem(cfg)

        from notebooks.agent.tools.base import ToolResult
        # Simulate the two tool calls the agent made.
        await hs.trigger_after_tool("read_file", {"path": "a.py"},
                                     ToolResult(success=True, output="src"))
        await hs.trigger_after_tool("write_file", {"path": "a.py"},
                                     ToolResult(success=True, output=""))
        await hs.trigger_after_tool("edit_file", {"path": "a.py"},
                                     ToolResult(success=True, output=""))

        print(open(trace).read() if trace.exists() else "(no lint fired)")

asyncio.run(lint_hook_demo())

**The hook fired exactly twice** — once for `write_file`, once for `edit_file` — and stayed silent for `read_file`. This is the entire appeal of hooks: behavior the agent itself does not and should not know about (`ruff check`, `npm test`, `git status`) can be added per project via a config file, with no changes to the agent's Python source.

:::{.callout-warning}
Hook scripts run with the *full* ambient environment (the env is `os.environ.copy()` plus the `AI_AGENT_*` vars). A hook that echoes secrets to a log file would leak them. Keep hooks short, audit them like any shell script, and prefer `command = "tool ..."` over inline `script` for anything non-trivial so the script is a versioned file rather than a string in your TOML.

:::

## Timeouts and Failure Isolation


A misbehaving hook is the textbook way to brick an otherwise healthy agent. Two layers defend against that:

1. **`timeout_sec`** — every hook runs under `asyncio.wait_for(..., timeout)`. On timeout the subprocess group is `SIGKILL`-ed (the loader uses `start_new_session=True` so the whole process tree dies, not just the shell), and a WARNING is logged.
2. **Exception swallowing** — every `_run_hook` call is wrapped in a `try/except` that logs at WARNING and returns. A hook that `exit 1`s, fails to start, or throws in the temp-file writer cannot propagate into the agent loop.

In [ ]:
import asyncio, tempfile
from pathlib import Path

async def timeout_demo():
    with tempfile.TemporaryDirectory() as d:
        cwd = Path(d)
        # Sleeps 5s, but the hook's timeout is 0.2s.
        hook = HookConfig(
            name="slow",
            trigger=HookTrigger.BEFORE_AGENT,
            command="sleep 5",
            timeout_sec=0.2,
        )
        cfg = Config(cwd=cwd, hooks_enabled=True, hooks=[hook])
        hs = HookSystem(cfg)

        import time
        t0 = time.monotonic()
        await hs.trigger_before_agent("go")
        print(f"elapsed: {time.monotonic() - t0:.2f}s  (capped by timeout_sec=0.2)")

        # A hook that exits non-zero is logged, not raised.
        bad = HookConfig(
            name="fails",
            trigger=HookTrigger.AFTER_AGENT,
            command="exit 7",
        )
        cfg2 = Config(cwd=cwd, hooks_enabled=True, hooks=[bad])
        hs2 = HookSystem(cfg2)
        await hs2.trigger_after_agent("go", "ok")
        print("non-zero hook did not propagate")

asyncio.run(timeout_demo())

**The elapsed time matches `timeout_sec`**, not `sleep 5`. The agent loop is never blocked for longer than the configured `timeout_sec` per hook, no matter what the hook does.

## Summary


| Layer | Role |
|---|---|
| `HookTrigger` (enum) | The five lifecycle points: `BEFORE_AGENT`, `AFTER_AGENT`, `BEFORE_TOOL`, `AFTER_TOOL`, `ON_ERROR` |
| `HookConfig` (Pydantic) | One hook: name, trigger, exactly one of `command` / `script`, `timeout_sec`, `enabled` |
| `Config.hooks_enabled` / `Config.hooks` | Global kill-switch + list of hook definitions |
| `HookSystem` | Filters enabled hooks at construction; runs each as a subprocess with `AI_AGENT_*` env vars |
| `Agent` integration | Calls the five `trigger_*` methods at the corresponding points in `run` and `_agentic_loop` |

: {tbl-colwidths="[25,75]"}

<br>

← [Configuration Loading](/notebooks/apps/cda/05-config.html) &emsp; → [GUI 1: Chat Interface](/notebooks/apps/cda/07-ui.html)

---

■